# Bernoulli and the Venturi tube

*fluids · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/bernoulli_venturi)

Water flowing through a narrowing pipe must speed up to carry the same flow rate ($A u = Q$), and by Bernoulli's law its pressure drops in the throat. Area, velocity and pressure are tensors over the pipe axis; a batch of tracer particles shows the speed-up.

**Vector form**

$$\nabla\cdot\mathbf v = 0,\qquad p + \tfrac12\rho\,\lVert\mathbf v\rVert^2 = \text{const along a streamline}$$

**Index (tensor) form**

$$A_k\,u_k = Q,\qquad p_k + \tfrac12\rho\,u_k^2 = p_1 + \tfrac12\rho\,u_1^2,\qquad \dot x_n = u(x_n)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_k$ | `axis(400, dev)` | (N,) | points along the pipe axis, index $k$ |
| $A_k = \pi r_k^2$ | `area(radius(x, r1, r2))` | (N,) | cross-section at every axis point |
| $u_k = Q / A_k$ | `velocity(Q, A)` | (N,) | continuity: the same volume per second passes every section |
| $p_k$ | `pressure(u, p1, u1, rho)` | (N,) | Bernoulli: faster flow ⇒ lower pressure |
| $x_n$ | `tracer_x` | (T, n) | positions of the batch of tracers, moved by RK4 |
| $Q$ | `flow['Q']` | scalar | volume flow rate $A_1 u_1$ in m³/s |

The flow is treated as one-dimensional: the velocity is uniform across each section and friction is ignored, which is what makes Bernoulli's law exact here. Tracers at a fixed fraction of the local radius follow the wall. The gauges are open tubes whose liquid column $h = p/(\rho g)$ grows with the local gauge pressure. Measuring $p_1 - p_2$ gives the flow rate, $Q = A_2\sqrt{2\Delta p/(\rho(1 - A_2^2/A_1^2))}$ — this is how a Venturi meter works. If the throat pressure falls below atmospheric, a real flow could cavitate; here a warning is printed instead.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/bernoulli_venturi/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import venturi_flow_rate

R1, RHO = 0.05, 1000.0  # inlet radius (m), water density (kg/m³)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    r2 = R1 / params["area_ratio"] ** 0.5  # throat radius from A1/A2
    p1 = params["p1"] * 1000  # inlet gauge pressure in Pa
    x = tensors.axis(400, dev)  # (N,)
    flow = solve.steady_flow(x, R1, r2, params["u1"], p1, RHO)  # dict of (N,) tensors
    x0, eta = tensors.tracers(7, 14, dev)  # (n,), (n,)
    times, tracer_x = solve.move_tracers(x0, R1, r2, flow["Q"], params["t_max"])

    u1, u2 = flow["u"][0].item(), flow["u"].max().item()
    dp = flow["p"][0].item() - flow["p"].min().item()
    A1, A2 = flow["A"][0].item(), flow["A"].min().item()
    print(f"flow rate Q = A1 u1 = {flow['Q'] * 1000:.3f} L/s")
    print(f"throat speed u2 = {u2:.3f} m/s,  u2/u1 = {u2 / u1:.3f},  A1/A2 = {A1 / A2:.3f}")
    print(f"pressure drop p1 - p2 = {dp:.0f} Pa  (Bernoulli: {0.5 * RHO * (u2**2 - u1**2):.0f} Pa)")
    print(f"Venturi meter: Q from the pressure drop = {venturi_flow_rate(dp, A1, A2, RHO) * 1000:.3f} L/s")
    if flow["p"].min() < 0:
        print("warning: the throat pressure is below atmospheric — raise p1 (a real throat could cavitate)")
    return plot.animate(x, flow, times, tracer_x, eta, R1, r2, out_dir / "bernoulli_venturi.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/bernoulli_venturi/equation.py
r"""Physics → tensor translation.

Steady, incompressible, frictionless flow through a pipe whose cross-section A(x) changes slowly.
Treating the flow as one-dimensional (velocity u(x) uniform over each cross-section):

    continuity (mass in = mass out):   A(x) u(x) = Q                       (Q = volume flow rate)
    Bernoulli along a streamline:      p(x) + ½ ρ u(x)^2 = p_1 + ½ ρ u_1^2

Every quantity is a tensor over the axis points x: shape (N,). Tracer particles simply ride
the local velocity, dx_n/dt = u(x_n), for a batch of n tracers at once.
"""

import math

import torch


def area(r: torch.Tensor) -> torch.Tensor:
    """Circular cross-section A = π r².   r: (N,) → (N,)"""
    return math.pi * r**2


def velocity(Q: float, A: torch.Tensor) -> torch.Tensor:
    """Continuity: u = Q / A.   (N,) → (N,)"""
    return Q / A


def pressure(u: torch.Tensor, p1: float, u1: float, rho: float) -> torch.Tensor:
    r"""Bernoulli: p = p_1 + ½ ρ (u_1² − u²).   (N,) → (N,)"""
    return p1 + 0.5 * rho * (u1**2 - u**2)


def venturi_flow_rate(dp: float, A1: float, A2: float, rho: float) -> float:
    r"""Invert the two laws (what a Venturi meter does): Q = A_2 \sqrt{2Δp / (ρ (1 − (A_2/A_1)^2))}."""
    return A2 * math.sqrt(2 * dp / (rho * (1 - (A2 / A1) ** 2)))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/bernoulli_venturi/tensors.py
"""Build the concrete tensors: the axis grid, the pipe radius r(x) and the tracer particles."""

import torch

LENGTH = 1.0  # pipe length (m); the profile repeats beyond it so tracers can loop around


def smoothstep(s: torch.Tensor) -> torch.Tensor:
    """0 for s ≤ 0, 1 for s ≥ 1, a smooth S-curve in between."""
    s = s.clamp(0.0, 1.0)
    return 3 * s**2 - 2 * s**3


def radius(x: torch.Tensor, r1: float, r2: float) -> torch.Tensor:
    """Pipe radius at any x: r1 at the inlet, narrowing to r2 in the throat, slowly opening again."""
    x = x % LENGTH
    narrowing = smoothstep((x - 0.15) / 0.22) * (1 - smoothstep((x - 0.55) / 0.3))  # 0 … 1 … 0
    return r1 - (r1 - r2) * narrowing


def axis(n: int, device: torch.device) -> torch.Tensor:
    """N points along the pipe axis: shape (N,)."""
    return torch.linspace(0.0, LENGTH, n, device=device)


def tracers(n_rows: int, n_cols: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Tracer start positions x (n,) and their fixed fractional heights η ∈ (−1, 1) (n,).

    A tracer at height η always sits at y = η r(x): in this one-dimensional picture it follows the wall shape.
    """
    eta = torch.linspace(-0.8, 0.8, n_rows, device=device)
    x = torch.arange(n_cols, device=device) * LENGTH / n_cols
    xx, ee = torch.meshgrid(x, eta, indexing="ij")  # (n_cols, n_rows)
    xx = xx + 0.5 * LENGTH / n_cols * (torch.arange(n_rows, device=device) % 2)  # stagger the rows
    return xx.flatten(), ee.flatten()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/bernoulli_venturi/solve.py
"""Evaluate the steady flow on the axis and move the tracers through it with RK4."""

import torch

from problems._common.integrators import integrate

from . import tensors
from .equation import area, pressure, velocity


def steady_flow(x: torch.Tensor, r1: float, r2: float, u1: float, p1: float, rho: float) -> dict:
    """Radius, area, velocity and pressure along the axis, all (N,), plus the flow rate Q."""
    r = tensors.radius(x, r1, r2)
    A = area(r)
    Q = u1 * area(torch.tensor(r1)).item()  # flow rate fixed by the inlet
    u = velocity(Q, A)
    return {"r": r, "A": A, "u": u, "p": pressure(u, p1, u1, rho), "Q": Q}


def move_tracers(
    x0: torch.Tensor, r1: float, r2: float, Q: float, t_max: float, dt: float = 0.01
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), positions (T, n)) for dx_n/dt = u(x_n) = Q / A(x_n), all tracers at once."""
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda x, t: velocity(Q, area(tensors.radius(x, r1, r2))), x0, dt, n_steps)
    times = torch.arange(n_steps + 1, device=x0.device) * dt
    return times, traj % tensors.LENGTH  # a tracer leaving at the outlet re-enters at the inlet


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/bernoulli_venturi/plot.py
"""Top: the Venturi tube with tracers and three pressure gauges. Bottom: u(x) and p(x) along the axis."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from . import tensors

GAUGES = [0.07, 0.46, 0.95]  # gauge positions along the axis (m): inlet, throat, outlet


def animate(
    x: torch.Tensor, flow: dict, times: torch.Tensor, tracer_x: torch.Tensor, eta: torch.Tensor, r1: float, r2: float,
    path: Path,
) -> Path:  # fmt: skip
    xs, r, u, p = (v.cpu().numpy() for v in (x, flow["r"], flow["u"], flow["p"]))
    t = times.cpu().numpy()
    px = tracer_x.cpu().numpy()  # (T, n)
    e = eta.cpu().numpy()  # (n,)
    p_kpa = p / 1000

    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7, 4.6), gridspec_kw={"height_ratios": [1.5, 1]}, sharex=True)
    ax.fill_between(xs, -r, r, color="#dbeafe")
    ax.plot(xs, r, color="#374151", lw=2)
    ax.plot(xs, -r, color="#374151", lw=2)
    # gauges: vertical glass tubes whose liquid column is proportional to the local pressure
    top = 2.8 * r1 / max(p_kpa.max(), 1e-9)  # plot height per kPa
    for i, xg in enumerate(GAUGES):
        k = int(np.argmin(abs(xs - xg)))
        h = max(p_kpa[k], 0.0) * top
        ax.add_patch(plt.Rectangle((xg - 0.012, r[k]), 0.024, h, color="#2563eb", alpha=0.8))
        ax.plot([xg - 0.012, xg - 0.012, xg + 0.012, xg + 0.012], [r[k], r[k] + 3.2 * r1, r[k] + 3.2 * r1, r[k]],
                color="#9ca3af", lw=1)  # fmt: skip
        side = 1 if i < len(GAUGES) - 1 else -1  # the last label goes to the left of its tube
        ax.text(
            xg + side * 0.02,
            r[k] + h,
            f"{p_kpa[k]:.1f} kPa",
            fontsize=8,
            va="center",
            ha="left" if side > 0 else "right",
        )
    dots = ax.scatter(px[0], e * tensors.radius(torch.tensor(px[0]), r1, r2).numpy(), s=8, color="#dc2626", zorder=3)
    ax.set_ylim(-1.3 * r1, 4.5 * r1)
    ax.set_ylabel("y (m)")
    ax.grid(False)
    clock = ax.set_title("Q")  # placeholder text so tight_layout leaves room

    ax2.plot(xs, u, color="#dc2626", lw=1.8)
    ax2.set_ylabel("u (m/s)", color="#dc2626")
    ax2.set_xlabel("x (m)")
    ax3 = ax2.twinx()
    ax3.plot(xs, p_kpa, color="#2563eb", lw=1.8)
    ax3.set_ylabel("p (kPa)", color="#2563eb")
    ax3.grid(False)

    def update(step: int):
        y = e * tensors.radius(torch.tensor(px[step]), r1, r2).numpy()
        dots.set_offsets(np.column_stack([px[step], y]))
        clock.set_text(f"Q = {flow['Q'] * 1000:.2f} L/s    t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path, fps=20)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'u1': 1.0,
    'area_ratio': 3.0,
    'p1': 10.0,
    't_max': 2.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.bernoulli_venturi")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.bernoulli_venturi.main")
problem_dir = Path("problems/fluids/bernoulli_venturi")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
